# Construcción del corpus mediante la hemeroteca de *El País*

Este notebook implementa el proceso de recopilación del corpus de estudio a partir de la hemeroteca digital de *El País*. El procedimiento se realiza de forma semiautomática mediante la selección manual de las páginas de resultados y consta de las siguientes etapas:

1. Recuperación de los artículos correspondientes a una página concreta de la hemeroteca.
2. Filtrado de las noticias publicadas entre el 17 de diciembre de 2010 y el 31 de diciembre de 2011, periodo de estudio de la investigación.
3. Extracción del cuerpo completo de los artículos que aún no forman parte del corpus.
4. Incorporación de los nuevos registros a un único archivo acumulativo:

`data/corpus_elpais_manual.csv`

El corpus se actualiza de forma incremental, evitando la duplicación de artículos ya descargados. Los posibles errores detectados durante la extracción se muestran únicamente en la consola con fines de supervisión y no se almacenan en archivos independientes.

In [10]:
# 1. INSTALAR TRAFILATURA

%pip install -q -U trafilatura


Note: you may need to restart the kernel to use updated packages.


In [72]:
# 2. LIBRERÍAS

import html
import json
import re
import time
import random

from pathlib import Path

import pandas as pd
import requests
import trafilatura

from bs4 import BeautifulSoup


In [ ]:
# 3. CONFIGURACIÓN GENERAL

FECHA_INICIO_DT = pd.Timestamp("2010-12-17")
FECHA_FIN_DT = pd.Timestamp("2011-12-31")

# Máximo de cuerpos que se descargarán en cada ejecución.
# Al volver a ejecutar, se saltarán los ya guardados
# y se descargarán los 3 siguientes.
MAX_ARTICULOS_POR_EJECUCION = 2

# Pausa aleatoria entre artículos.
PAUSA_MINIMA = 30
PAUSA_MAXIMA = 90

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# Único CSV que utilizará el notebook.
RUTA_CORPUS = DATA_DIR / "corpus_elpais_manual.csv"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/138.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "es-ES,es;q=0.9",
}

SESSION = requests.Session()
SESSION.headers.update(HEADERS)

print("CSV acumulativo:", RUTA_CORPUS.resolve())
print(
    "Máximo por ejecución:",
    MAX_ARTICULOS_POR_EJECUCION
)


CSV acumulativo: /Users/maca/Documents/Academico/IT academy cibernarium/Análisis de datos/Especialización/Proyecto final/data/corpus_elpais_manual.csv
Máximo por ejecución: 2


## 4. Página manual actual


In [ ]:
# 4. PÁGINA MANUAL ACTUAL

PALABRA_CLAVE = "Túnez"

URL_PAGINA_HEMEROTECA = (
    "https://elpais.com/noticias/tunez/1/"
)

NUMERO_PAGINA = 1


In [111]:
# 5. FUNCIONES AUXILIARES

def limpiar_texto(texto):
    if texto is None:
        return None

    texto = html.unescape(str(texto))
    texto = re.sub(r"\s+", " ", texto).strip()

    return texto or None


def normalizar_url(url):
    if not isinstance(url, str):
        return None

    return (
        url.split("#")[0]
        .split("?")[0]
        .rstrip("/")
    )


def leer_corpus():
    """Lee el CSV acumulativo o devuelve un DataFrame vacío."""

    if not RUTA_CORPUS.exists():
        return pd.DataFrame()

    if RUTA_CORPUS.stat().st_size == 0:
        return pd.DataFrame()

    try:
        return pd.read_csv(
            RUTA_CORPUS,
            encoding="utf-8-sig"
        )

    except pd.errors.EmptyDataError:
        return pd.DataFrame()


def guardar_corpus(df):
    df.to_csv(
        RUTA_CORPUS,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"✔ Guardado: {RUTA_CORPUS.resolve()}")


In [ ]:
# 6. EXTRAER REFERENCIAS DE UNA SOLA PÁGINA

def extraer_info_pagina(
    url_pagina,
    palabra_clave,
    numero_pagina
):
    """
    Hace una sola petición a la página concreta de la hemeroteca.
    """

    try:
        response = SESSION.get(
            url_pagina,
            timeout=30
        )

    except requests.RequestException as error:
        print(
            "Error de conexión en la hemeroteca:",
            f"{type(error).__name__}: {error}"
        )
        return None

    if response.status_code != 200:
        print(
            f"Error HTTP {response.status_code} "
            "al descargar la hemeroteca."
        )
        return None

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    resultados = []

    for posicion, articulo in enumerate(
        soup.find_all("article"),
        start=1
    ):
        texto_completo = articulo.get_text(
            " ",
            strip=True
        )

        coincidencia_fecha = re.search(
            r"\b\d{2}/\d{2}/\d{4}\b",
            texto_completo
        )

        fecha = (
            coincidencia_fecha.group()
            if coincidencia_fecha
            else None
        )

        hora = None
        zona_horaria = None

        for span in articulo.find_all("span"):
            texto_span = span.get_text(
                " ",
                strip=True
            )

            coincidencia_hora = re.search(
                r"\b\d{2}:\d{2}\b",
                texto_span
            )

            if coincidencia_hora:
                hora = coincidencia_hora.group()

                coincidencia_zona = re.search(
                    r"\b(?:CET|CEST)\b",
                    texto_span
                )

                if coincidencia_zona:
                    zona_horaria = coincidencia_zona.group()

                break

        elemento_antetitulo = articulo.select_one("a.c_k")
        antetitulo = (
            elemento_antetitulo.get_text(" ", strip=True)
            if elemento_antetitulo
            else None
        )

        elemento_titulo = articulo.select_one("h2.c_t")

        titulo = None
        url = None

        if elemento_titulo:
            titulo = elemento_titulo.get_text(
                " ",
                strip=True
            )

            enlace_titulo = elemento_titulo.find(
                "a",
                href=True
            )

            if enlace_titulo:
                url = enlace_titulo.get("href")

        if titulo and not url:
            for enlace in articulo.find_all(
                "a",
                href=True
            ):
                if enlace.get_text(" ", strip=True) == titulo:
                    url = enlace.get("href")
                    break

        elemento_autor = articulo.select_one("a.c_a_a")
        autor = (
            elemento_autor.get_text(" ", strip=True)
            if elemento_autor
            else None
        )

        elemento_subtitulo = articulo.select_one("p.c_d")
        subtitulo = (
            elemento_subtitulo.get_text(" ", strip=True)
            if elemento_subtitulo
            else None
        )

        if not titulo or not url:
            continue

        fecha_dt = pd.to_datetime(
            fecha,
            format="%d/%m/%Y",
            errors="coerce"
        )

        resultados.append({
            "fecha": fecha,
            "hora": hora,
            "zona_horaria": zona_horaria,
            "antetitulo": antetitulo,
            "titulo": titulo,
            "autor": autor,
            "subtitulo": subtitulo,
            "url": url,
            "palabra_clave": palabra_clave,
            "pagina_busqueda": numero_pagina,
            "posicion": posicion,
            "fecha_dt": fecha_dt,
        })

    print(
        "Resultados encontrados en la página:",
        len(resultados))
    
    return resultados


In [113]:
# 7. EXTRAER EL CUERPO DE UNA NOTICIA

def encontrar_json_ld_articulo(soup):
    tipos_validos = {
        "NewsArticle",
        "Article",
        "ReportageNewsArticle",
        "AnalysisNewsArticle",
        "OpinionNewsArticle",
    }

    pendientes = []

    for script in soup.find_all(
        "script",
        attrs={"type": "application/ld+json"}
    ):
        contenido = (
            script.string
            or script.get_text(strip=True)
        )

        if not contenido:
            continue

        try:
            pendientes.append(json.loads(contenido))
        except (json.JSONDecodeError, TypeError):
            continue

    while pendientes:
        elemento = pendientes.pop()

        if isinstance(elemento, list):
            pendientes.extend(elemento)
            continue

        if not isinstance(elemento, dict):
            continue

        tipo = elemento.get("@type")

        if (
            tipo in tipos_validos
            or (
                isinstance(tipo, list)
                and tipos_validos.intersection(tipo)
            )
        ):
            return elemento

        for valor in elemento.values():
            if isinstance(valor, (dict, list)):
                pendientes.append(valor)

    return {}


def extraer_cuerpo_noticia(url):
    """
    Hace una sola petición a la noticia.

    Devuelve
    --------
    tuple
        cuerpo, error, status_code
    """

    try:
        response = SESSION.get(
            url,
            timeout=30,
            allow_redirects=True
        )

    except requests.RequestException as error:
        return (
            None,
            f"{type(error).__name__}: {error}",
            None
        )

    if response.status_code != 200:
        return (
            None,
            f"HTTP {response.status_code}",
            response.status_code
        )

    documento = response.text
    cuerpo = None

    salida_json = trafilatura.extract(
        documento,
        url=response.url,
        output_format="json",
        with_metadata=True,
        include_comments=False,
        include_tables=False,
        favor_recall=True,
    )

    if salida_json:
        try:
            datos = json.loads(salida_json)

            cuerpo = limpiar_texto(
                datos.get("text")
                or datos.get("raw_text")
            )

        except json.JSONDecodeError:
            pass

    if not cuerpo:
        soup = BeautifulSoup(
            documento,
            "html.parser"
        )

        articulo_ld = encontrar_json_ld_articulo(soup)

        cuerpo = limpiar_texto(
            articulo_ld.get("articleBody")
        )

    if not cuerpo:
        soup = BeautifulSoup(
            documento,
            "html.parser"
        )

        selectores = [
            "[data-dtm-region='articulo_cuerpo'] p",
            "[itemprop='articleBody'] p",
            "article p",
        ]

        for selector in selectores:
            parrafos = []

            for elemento in soup.select(selector):
                texto = limpiar_texto(
                    elemento.get_text(" ", strip=True)
                )

                if texto and len(texto) >= 40:
                    parrafos.append(texto)

            parrafos = list(dict.fromkeys(parrafos))

            if parrafos:
                cuerpo = "\n\n".join(parrafos)
                break

    if not cuerpo:
        return (
            None,
            "Cuerpo no localizado",
            response.status_code
        )

    return cuerpo, None, response.status_code


In [114]:
# 8. CARGAR EL ÚNICO CSV Y DETECTAR URLs YA GUARDADAS

df_acumulado = leer_corpus()

urls_guardadas = set()

if (
    not df_acumulado.empty
    and "url" in df_acumulado.columns
):
    urls_guardadas.update(
        df_acumulado["url"]
        .dropna()
        .map(normalizar_url)
    )

print(
    "Artículos ya guardados:",
    len(urls_guardadas)
)


Artículos ya guardados: 138


In [115]:
# 9. EXTRAER LA PÁGINA MANUAL Y SELECCIONAR ARTÍCULOS NUEVOS

resultados_pagina = extraer_info_pagina(
    url_pagina=URL_PAGINA_HEMEROTECA,
    palabra_clave=PALABRA_CLAVE,
    numero_pagina=NUMERO_PAGINA
)

if resultados_pagina is None:
    raise RuntimeError(
        "No se ha podido descargar la página de hemeroteca."
    )

df_pagina = pd.DataFrame(resultados_pagina)

if not df_pagina.empty:
    df_pagina["fecha_dt"] = pd.to_datetime(
        df_pagina["fecha"],
        format="%d/%m/%Y",
        errors="coerce"
    )

    df_pagina_filtrada = df_pagina[
        df_pagina["fecha_dt"].between(
            FECHA_INICIO_DT,
            FECHA_FIN_DT,
            inclusive="both"
        )
    ].copy()

    df_pagina_filtrada["url_normalizada"] = (
        df_pagina_filtrada["url"].map(normalizar_url)
    )

    df_nuevos = df_pagina_filtrada[
        ~df_pagina_filtrada["url_normalizada"]
        .isin(urls_guardadas)
    ].copy()

    # Procesar solo los primeros artículos pendientes.
    if MAX_ARTICULOS_POR_EJECUCION is not None:
        df_nuevos = df_nuevos.head(
            MAX_ARTICULOS_POR_EJECUCION
        ).copy()

else:
    df_pagina_filtrada = pd.DataFrame()
    df_nuevos = pd.DataFrame()

print(
    "Resultados dentro del periodo:",
    len(df_pagina_filtrada)
)

print(
    "Artículos pendientes seleccionados para esta ejecución:",
    len(df_nuevos)
)

print(
    "Límite configurado:",
    MAX_ARTICULOS_POR_EJECUCION
)


Error de conexión en la hemeroteca: ConnectionError: HTTPSConnectionPool(host='elpais.com', port=443): Max retries exceeded with url: /noticias/primavera-arabe/1/ (Caused by NameResolutionError("HTTPSConnection(host='elpais.com', port=443): Failed to resolve 'elpais.com' ([Errno 8] nodename nor servname provided, or not known)"))


RuntimeError: No se ha podido descargar la página de hemeroteca.

In [ ]:
# 10. DESCARGAR CUERPOS Y ACTUALIZAR EL ÚNICO CSV

nuevos_registros = []
bloqueo_403 = False

total = len(df_nuevos)

for numero, fila in enumerate(
    df_nuevos.itertuples(index=False),
    start=1
):
    print(
        f"{numero}/{total} | {fila.titulo}"
    )

    cuerpo, error, status_code = (
        extraer_cuerpo_noticia(fila.url)
    )

    # Si aparece un 403, detener inmediatamente.
    if status_code == 403:
        print(
            "  ✘ HTTP 403. "
            "Se detiene esta ejecución para no insistir."
        )

        bloqueo_403 = True
        break

    if cuerpo is None:
        print("  ✘", error)

    else:
        registro = fila._asdict()

        registro.pop("url_normalizada", None)

        registro["cuerpo_noticia"] = cuerpo
        registro["longitud_cuerpo"] = len(cuerpo)

        nuevos_registros.append(registro)

        print(
            "  ✔",
            len(cuerpo),
            "caracteres"
        )

        # Guardar inmediatamente tras cada artículo correcto.
        partes = [
            df for df in [
                df_acumulado,
                pd.DataFrame(nuevos_registros)
            ]
            if not df.empty
        ]

        df_actual = pd.concat(
            partes,
            ignore_index=True
        )

        df_actual = (
            df_actual
            .drop_duplicates(
                subset="url",
                keep="first"
            )
            .reset_index(drop=True)
        )

        guardar_corpus(df_actual)

    # Pausa aleatoria antes del siguiente artículo.
    if numero < total and not bloqueo_403:
        pausa = random.randint(
            PAUSA_MINIMA,
            PAUSA_MAXIMA
        )

        print(
            f"  Pausa de {pausa} segundos..."
        )

        time.sleep(pausa)

if bloqueo_403:
    print(
        "\nEjecución detenida por 403. "
        "Los artículos correctos anteriores ya están guardados."
    )


1/1 | La «calle» tunecina
  ✔ 4816 caracteres
✔ Guardado: /Users/maca/Documents/Academico/IT academy cibernarium/Análisis de datos/Especialización/Proyecto final/data/corpus_elpais_manual.csv


In [ ]:
# 11. RESULTADO FINAL

partes = [
    df for df in [
        df_acumulado,
        pd.DataFrame(nuevos_registros)
    ]
    if not df.empty
]

df_corpus_final = (
    pd.concat(partes, ignore_index=True)
    if partes
    else pd.DataFrame()
)

if not df_corpus_final.empty:
    df_corpus_final = (
        df_corpus_final
        .drop_duplicates(
            subset="url",
            keep="first"
        )
        .reset_index(drop=True)
    )

    columnas_finales = [
        "fecha",
        "hora",
        "zona_horaria",
        "antetitulo",
        "titulo",
        "autor",
        "subtitulo",
        "url",
        "palabra_clave",
        "pagina_busqueda",
        "posicion",
        "fecha_dt",
        "cuerpo_noticia",
        "longitud_cuerpo",
    ]

    columnas_existentes = [
        columna
        for columna in columnas_finales
        if columna in df_corpus_final.columns
    ]

    df_corpus_final = df_corpus_final[
        columnas_existentes
    ]

    guardar_corpus(df_corpus_final)

print(
    "Artículos acumulados:",
    len(df_corpus_final)
)

if not df_corpus_final.empty:
    display(
        df_corpus_final[
            [
                "fecha",
                "titulo",
                "autor",
                "url",
                "cuerpo_noticia",
            ]
        ].tail(20)
    )


✔ Guardado: /Users/maca/Documents/Academico/IT academy cibernarium/Análisis de datos/Especialización/Proyecto final/data/corpus_elpais_manual.csv
Artículos acumulados: 138


,fecha,titulo,autor,url,cuerpo_noticia
118,19/10/2011,Los islamistas se perfilan como ganadores en T...,Ignacio Cembrero,https://elpais.com/internacional/2011/10/19/ac...,Los islamistas se perfilan como ganadores en T...
119,18/11/2011,Las provocaciones islamistas inquietan a los l...,Ignacio Cembrero,https://elpais.com/internacional/2011/11/18/ac...,Las provocaciones islamistas inquietan a los l...
120,01/11/2011,Islam y democracia,Antonio Elorza,https://elpais.com/diario/2011/11/01/opinion/1...,Islam y democracia El humorista del diario tun...
121,31/10/2011,Islam y democracia,Antonio Elorza,https://elpais.com/internacional/2011/10/31/ac...,Islam y democracia El nuevo régimen de Túnez p...
122,07/02/2011,El islam y la democracia,Cartas al Director,https://elpais.com/diario/2011/02/07/opinion/1...,El islam y la democracia Ante quienes recurren...
123,15/01/2011,El avión tunecino que aterrizó en Cerdeña aban...,NaN,https://elpais.com/internacional/2011/01/15/ac...,El avión tunecino que aterrizó en Cerdeña aban...
124,28/01/2011,El mundo árabe empieza a notar el efecto conta...,NaN,https://elpais.com/internacional/2011/01/28/ac...,El mundo árabe empieza a notar el efecto conta...
125,16/01/2011,Muere supuestamente apuñalado el sobrino de la...,NaN,https://elpais.com/internacional/2011/01/16/ac...,Muere supuestamente apuñalado el sobrino de la...
126,15/01/2011,"Día de miedo, saqueos y disturbios en Túnez",NaN,https://elpais.com/internacional/2011/01/15/ac...,"Día de miedo, saqueos y disturbios en Túnez De..."
127,14/02/2011,Continúa la llegada de inmigrantes tunecinos a...,NaN,https://elpais.com/internacional/2011/02/14/ac...,Continúa la llegada de inmigrantes tunecinos a...
